In [1]:
!git clone https://github.com/Mr-Gump/seq-model.git
!pip install -r seq-model/requirements.txt
!wget https://github.com/amphibian-dev/toad/archive/refs/tags/0.1.5.tar.gz
!tar -zxvf 0.1.5.tar.gz
!cd toad-0.1.5 && python setup.py install

Cloning into 'seq-model'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 63 (delta 22), reused 45 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 579.36 KiB | 4.36 MiB/s, done.
Resolving deltas: 100% (22/22), done.
--2026-04-07 08:36:08--  https://github.com/amphibian-dev/toad/archive/refs/tags/0.1.5.tar.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/amphibian-dev/toad/tar.gz/refs/tags/0.1.5 [following]
--2026-04-07 08:36:08--  https://codeload.github.com/amphibian-dev/toad/tar.gz/refs/tags/0.1.5
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
L

# 需要重启 kernel

In [2]:
import os
import sys

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = '/content/drive/MyDrive/dataset/seq-data/'
    sys.path.append("/content/seq-model")
else:
    DATASET_DIR = './dataset'

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd
import os

seq_data_path = os.path.join(DATASET_DIR, 'event_lst.csv')
sample_path = os.path.join(DATASET_DIR, 'tmp_0403.csv')

df = pd.read_csv(seq_data_path)
df['event_lst'] = df['event_lst'].apply(eval)
df['event_num'] = df['event_lst'].apply(len)

In [3]:
df_sample = pd.read_csv(sample_path)

In [4]:
df.drop_duplicates(subset=['sn'], inplace=True)
df_sample.drop_duplicates(subset=['sn'], inplace=True)

In [5]:
def get_event_seq(event_lst):
    return [event['event_id'] for event in event_lst]

def get_time_delta_seq(event_lst):
    return [event['time_delta'] for event in event_lst]

def get_value_seq(event_lst):
    return [
        (event['onway_amt'], event['onway_cnt'], event['borrow_amt'], event['is_same_pkg'])
        for event in event_lst
    ]

In [6]:
df['event_seq'] = df['event_lst'].apply(get_event_seq)
df['time_delta_seq'] = df['event_lst'].apply(get_time_delta_seq)
df['value_seq'] = df['event_lst'].apply(get_value_seq)

## 针对 log1p 压缩特性，将时间差序列转换为剩余时间序列，以更好地捕捉时间信息

In [7]:
import numpy as np

def handle_time_delta_seq(time_delta_seq):
    sum = 0
    total_sum = np.sum(time_delta_seq)
    for i in range(len(time_delta_seq)):
        sum += time_delta_seq[i]
        time_delta_seq[i] = total_sum - sum
    return time_delta_seq

df['time_delta_seq'] = df['time_delta_seq'].apply(handle_time_delta_seq)

In [8]:
df_merge = pd.merge(df_sample, df[['sn', 'event_seq','time_delta_seq','value_seq']], on='sn', how='inner')

In [9]:
df_merge

,sn,verify_time,target,reloan_score_2508_score,multi_borrow_count,global_borrow_count,event_seq,time_delta_seq,value_seq
0,CMS2026020910083479169892,1770606649,0,617.400262,3,2,"[1, 1, 5, 8, 1, 5, 8]","[3024168, 1191224, 1191134, 784507, 472485, 47...","[(0, 0, 0, 1), (0, 0, 0, 1), (0, 0, 4500.0, 1)..."
1,CMS2026022110565190425930,1771646230,0,629.405405,4,3,"[1, 1, 5, 8, 1, 5, 8, 1, 5, 7]","[57140470, 55307526, 55307436, 54900809, 54588...","[(0, 0, 0, 1), (0, 0, 0, 1), (0, 0, 4500.0, 1)..."
2,EDD20251130044636kYkJ36g7,1764496001,0,628.079043,1,3,"[1, 5, 1, 5, 8, 8, 1]","[871242, 871137, 859766, 859723, 443007, 44284...","[(0, 0, 0, 0), (0, 0, 3300.0, 0), (3300.0, 1, ..."
3,EDD20251130074911MNREI36Z,1764506956,1,580.394978,7,32,"[1, 1, 5, 8, 1, 5, 11, 10, 1, 1, 5, 1, 1, 1, 5...","[52501806, 44202414, 44202392, 43712039, 43369...","[(0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 2500.0, 0)..."
4,EDD20251130075400Jb6GcO7n,1764507245,1,604.912102,1,2,"[1, 5, 1, 5, 8, 8, 1]","[687597, 687539, 409833, 409772, 249179, 61, 0]","[(0, 0, 0, 1), (0, 0, 1800.0, 1), (1800.0, 1, ..."
...,...,...,...,...,...,...,...,...,...
98648,SVD20260301042307y5DvST74,1772356990,0,618.590732,34,34,"[1, 5, 5, 8, 8, 1, 5, 5, 8, 8, 1, 5, 7, 1, 5, ...","[14082970, 14082927, 14082892, 13675418, 13663...","[(0, 0, 0, 1), (0, 0, 3500.0, 1), (3500.0, 1, ..."
98649,SVD20260301064041HQ09RaY7,1772365244,0,598.078854,5,10,"[1, 5, 8, 1, 5, 4, 1, 5, 4, 4, 4, 11, 12, 9, 1...","[25860105, 25859939, 25426689, 24634594, 24634...","[(0, 0, 0, 0), (0, 0, 5000.0, 0), (5000.0, 1, ..."
98650,SVD2026030109320524EP5K5W,1772375529,0,590.785487,10,22,"[3, 5, 9, 1, 1, 5, 1, 5, 1, 5, 9, 9, 8, 1, 1, ...","[19042750, 19012081, 18484668, 18484406, 18435...","[(0, 0, 0, 0), (0, 0, 2700.0, 0), (2700.0, 1, ..."
98651,SVD20260301100941TW655kAf,1772334587,0,615.874396,13,64,"[1, 2, 2, 1, 1, 5, 8, 1, 5, 8, 1, 5, 8, 1, 5, ...","[59594476, 59510857, 59430440, 57005522, 56569...","[(0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,..."


In [10]:
from sklearn.model_selection import train_test_split

df_merge = df_merge.sort_values('verify_time').reset_index(drop=True)

n = len(df_merge)
oot_start = int(n * 0.8)

df_in_time = df_merge.iloc[:oot_start]
df_oot = df_merge.iloc[oot_start:]

# 同一时间段内随机划分训练集(5/8)和测试集(3/8)，即总体 50%:30%
df_train, df_test = train_test_split(df_in_time, test_size=3/8, random_state=42, stratify=df_in_time['target'])

print(f"训练集: {len(df_train)} ({len(df_train)/n:.1%}), 日期范围: {df_train['verify_time'].min()} ~ {df_train['verify_time'].max()}")
print(f"测试集: {len(df_test)} ({len(df_test)/n:.1%}), 日期范围: {df_test['verify_time'].min()} ~ {df_test['verify_time'].max()}")
print(f"验证集(OOT): {len(df_oot)} ({len(df_oot)/n:.1%}), 日期范围: {df_oot['verify_time'].min()} ~ {df_oot['verify_time'].max()}")
print(f"\ntarget分布:")
print(f"  训练集: {df_train['target'].mean():.4f}")
print(f"  测试集: {df_test['target'].mean():.4f}")
print(f"  验证集: {df_oot['target'].mean():.4f}")

训练集: 49326 (50.0%), 日期范围: 1764496001 ~ 1771106923
测试集: 29596 (30.0%), 日期范围: 1764506956 ~ 1771106822
验证集(OOT): 19731 (20.0%), 日期范围: 1771107581 ~ 1772382556

target分布:
  训练集: 0.2165
  测试集: 0.2165
  验证集: 0.2357


In [11]:
from event_classifier_v2.main import main
import warnings

warnings.filterwarnings("ignore")

In [ ]:
CFG = dict(
    max_seq_len = 128,
    batch_size  = 128,
    n_epochs    = 20,
    lr          = 1e-3,
    weight_decay= 1e-2,
    save_dir    = "./checkpoints",
    # 模型结构
    num_event_types = 13,
    d_model     = 48,
    d_event     = 32,
    d_time      = 32,
    d_cont      = 12,   # 连续特征：onway_amt, onway_cnt, borrow_amt
    d_pkg       = 4,   # is_same_pkg embedding
    n_heads     = 4,
    n_layers    = 2,
    d_ffn       = 72,
    dropout     = 0.1,
)

In [12]:
df_out = main(df_train, df_test, df_oot, CFG)

训练集 | 总量:49326  好:38647  坏:10679  坏率:21.65%  pos_weight:3.62
测试集 | 总量:29596   坏率:21.65%
OOT集  | 总量:19731    坏率:23.57%

模型参数量：35,129

Device: cuda  |  pos_weight: 3.62

 Epoch │   TrLoss   TrAUC    TrKS │  ValLoss  ValAUC   ValKS   ValPR
──────────────────────────────────────────────────────────────────────
     1 │   1.0776  0.5763  0.1136 │   1.0423  0.6374  0.2005  0.3139 ◀ best
     2 │   1.0489  0.6232  0.1783 │   1.0377  0.6481  0.2050  0.3291 ◀ best
     3 │   1.0378  0.6405  0.2039 │   1.0202  0.6626  0.2321  0.3442 ◀ best
     4 │   1.0287  0.6535  0.2249 │   1.0412  0.6703  0.2414  0.3559 ◀ best
     5 │   1.0174  0.6662  0.2405 │   1.0135  0.6778  0.2549  0.3637 ◀ best
     6 │   1.0104  0.6748  0.2562 │   1.0159  0.6761  0.2480  0.3594
     7 │   1.0070  0.6773  0.2569 │   1.0082  0.6790  0.2603  0.3638 ◀ best
     8 │   1.0042  0.6801  0.2595 │   1.0078  0.6804  0.2600  0.3671
     9 │   1.0021  0.6820  0.2630 │   1.0040  0.6814  0.2650  0.3665 ◀ best
    10 │   0.9993  0.6

In [13]:
df_test_res,df_oot_res = df_out[1],df_out[2]

In [14]:
def prob_to_score(prob):
    factor = 20 / np.log(2)
    offset = 600 + factor * np.log(0.284)
    odds = prob / (1 - prob)
    score = offset - factor * np.log(odds)
    return score

In [15]:
df_test_res['score'] = df_test_res['prob'].apply(prob_to_score)
df_oot_res['score'] = df_oot_res['prob'].apply(prob_to_score)

In [ ]:
df_test_res = df_test_res.merge(df_sample[['sn','reloan_score_2508_score']],how='inner',on='sn')
df_oot_res = df_oot_res.merge(df_sample[['sn','reloan_score_2508_score']],how='inner',on='sn')

In [16]:
import toad

bucket = toad.metrics.KS_bucket(df_oot_res['score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

,min,max,bads,goods,total,bad_rate,good_rate,odds,bad_prop,good_prop,...,cum_bad_rate_rev,cum_bads_prop,cum_bads_prop_rev,cum_goods_prop,cum_goods_prop_rev,cum_total_prop,cum_total_prop_rev,ks,lift,cum_lift
0,499.767735,533.838166,919,1054,1973,0.465788,0.534212,0.871917,0.197592,0.069894,...,0.235720,0.197592,1.000000,0.069894,1.000000,0.099995,1.000000,0.127698,1.976019,1.976019
1,533.839348,543.304496,661,1312,1973,0.335023,0.664977,0.503811,0.142120,0.087003,...,0.210159,0.339712,0.802408,0.156897,0.930106,0.199990,0.900005,0.182815,1.421272,1.698646
2,543.308685,550.349831,601,1372,1973,0.304612,0.695388,0.438047,0.129220,0.090981,...,0.194552,0.468931,0.660288,0.247878,0.843103,0.299985,0.800010,0.221053,1.292261,1.563184
3,550.350977,556.144085,510,1463,1973,0.258490,0.741510,0.348599,0.109654,0.097016,...,0.178830,0.578585,0.531069,0.344894,0.752122,0.399980,0.700015,0.233691,1.096594,1.446536
4,556.147563,561.557707,497,1476,1973,0.251901,0.748099,0.336721,0.106859,0.097878,...,0.165555,0.685444,0.421415,0.442772,0.655106,0.499975,0.600020,0.242672,1.068642,1.370957
5,561.558665,566.258080,430,1543,1973,0.217942,0.782058,0.278678,0.092453,0.102321,...,0.148287,0.777897,0.314556,0.545093,0.557228,0.599970,0.500025,0.232804,0.924579,1.296561
6,566.258539,570.470154,343,1630,1973,0.173847,0.826153,0.210429,0.073748,0.108090,...,0.130875,0.851645,0.222103,0.653183,0.454907,0.699965,0.400030,0.198462,0.737513,1.216697
7,570.471664,577.861581,330,1643,1973,0.167258,0.832742,0.200852,0.070952,0.108952,...,0.116554,0.922597,0.148355,0.762135,0.346817,0.799959,0.300035,0.160462,0.709561,1.153305
8,577.875287,593.394928,231,1742,1973,0.117081,0.882919,0.132606,0.049667,0.115517,...,0.091209,0.972264,0.077403,0.877653,0.237865,0.899954,0.200041,0.094612,0.496693,1.080348
9,593.397270,638.230099,129,1845,1974,0.065350,0.934650,0.069919,0.027736,0.122347,...,0.065350,1.000000,0.027736,1.000000,0.122347,1.000000,0.100046,0.000000,0.277233,1.000000


In [17]:
bucket = toad.metrics.KS_bucket(df_oot_res['reloan_score_2508_score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

KeyError: 'reloan_score_2508_score'

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df_test_res[['score', 'reloan_score_2508_score']].corr(), annot=True, cmap='RdBu_r', vmin=-1, vmax=1)
plt.show()

In [ ]:
# 1. 分别等频十分箱
df_oot_res['score_bin'] = pd.qcut(df_oot_res['score'], q=10, duplicates='drop')
df_oot_res['reloan_score_2508_score_bin'] = pd.qcut(df_oot_res['reloan_score_2508_score'], q=10, duplicates='drop')

# 2. 交叉统计 target 浓度（坏样本率）
cross = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='mean'
)

# 3. 交叉统计每个格子的样本量（辅助参考）
cross_count = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='count'
)

# 4. 热力图可视化
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cross, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('bad_rate')
axes[0].set_xlabel('reloan_score_2508_score_bin')
axes[0].set_ylabel('score_bin')

sns.heatmap(cross_count, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('loan_cnt')
axes[1].set_xlabel('reloan_score_2508_score_bin')
axes[1].set_ylabel('score_bin')

plt.tight_layout()
plt.show()